In [4]:
# !pip install scikit-learn -q
# !pip install category-encoders -q
# !pip install seaborn -q
# !pip install openpyxl -q

In [5]:
import tinkoffpy as tf
from getpass import getpass
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [6]:
# tf.auth_by_credentials(
#     username='a.p.burov',
#     password=getpass("Password:")
# )

# gp = tf.Greenplum()

In [7]:
# df = gp.gp_to_df(
#     sql_query='select * from usr_oper.ml_table_classification_case',
#     gp_service='orig'
# )

df = pd.read_excel('https://s3-msk.tinkoff.ru/data-growth-feature-factory/ml4pa/case_dataset.xlsx', sheet_name=0)

df

HTTPError: HTTP Error 403: Forbidden

# EDA

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps
import seaborn as sns

In [ ]:
## Смотрим общую информацию о датасете

df.info()

In [ ]:
## Смотрим колчество наблюдений для каждого класса

df.target.value_counts().plot(kind='bar', title='Баланс классов')

In [ ]:
## Смотрим на хиты фичей на нашей выборке

hit_data = df.describe().loc['count'] / df.shape[0]
hit_data_values = hit_data.values.astype(float)

cmap = colormaps.get_cmap('RdYlGn')
colors = cmap(hit_data_values)

hit_data_series = pd.Series(hit_data_values, index=hit_data.index)

plt.figure(figsize=(20, 8))
hit_data_series.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Хиты фичей')
plt.xlabel('Фича')
plt.ylabel('Хит')
plt.xticks(rotation=90)
plt.show()

In [ ]:
## Смотрим на категориальные фичи
cat_features = df.select_dtypes(include=['object', 'category']).columns.to_list()

cat_features

# Предобработка

In [ ]:
keys = ['party_rk', 'party_dttm'] # Ключи обычно не используются как фичи в модели
df.drop(columns=keys, inplace=True)

df

In [ ]:
## Обновляем список категориальных фичей на всякий случай
cat_features = df.select_dtypes(include=['object', 'category']).columns.to_list()

cat_features

In [ ]:
df.drop(columns=cat_features, inplace=True)

df.shape

In [ ]:
## Удаляем столбцы, которые содержат пропуски

df.dropna(axis=1, inplace=True)

df

In [ ]:
from sklearn.model_selection import train_test_split


## Делим обучающую выборку на тренировочную и тестовую
## Размер тестовой выборки определяется параметром test_size = 0.25 (25% данных пойдет на тест)
X = df.drop(columns='target')
y = df.target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print('Размерность тренировочных данных:', X_train.shape, y_train.shape)
print('Размерность тестовых данных:', X_test.shape, y_test.shape)

# Обучение моделей

Обучите модель LogisticRegression из библиотеки sklean

Ссылка на документацию: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [ ]:
from sklearn.linear_model import LogisticRegression

... 
y_preds = ...
y_preds_proba = ...

## Оценка модели: метрики

### **Формат матрицы ошибок**

Для бинарной классификации (например, классы 0 и 1) матрица ошибок выглядит так:

\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}

Где:

- **TN (True Negative)** – верно предсказанные отрицательные примеры (0 → 0)
- **FP (False Positive)** – ложные срабатывания (ошибочно предсказан 1, но был 0)
- **FN (False Negative)** – пропущенные объекты (ошибочно предсказан 0, но был 1)
- **TP (True Positive)** – верно предсказанные положительные примеры (1 → 1)


Выведите матрицу ошибок с помощью **confusion_matrix** из **sklearn.metrics**

In [ ]:
from sklearn.metrics import confusion_matrix


cm = ...

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Предсказано 0", "Предсказано 1"],
            yticklabels=["Факт 0", "Факт 1"])
plt.title("Confusion Matrix")
plt.show()

### 🔹 Метрики качества классификации

#### ✅ **1. Accuracy (Точность классификации)**
Общая доля правильно классифицированных объектов:

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$
➡️ **Хороша при сбалансированных классах, но не подходит для несбалансированных данных.**

---

#### 🎯 **2. Precision (Точность)**
Доля правильных положительных предсказаний среди всех предсказанных положительных:

$$
Precision = \frac{TP}{TP + FP}
$$

➡️ **Важно в задачах, где ложные положительные предсказания критичны (например, диагностика рака).**

---

#### 🔄 **3. Recall (Полнота)**
Доля правильно предсказанных положительных примеров среди всех реальных положительных примеров:

$$
Recall = \frac{TP}{TP + FN}
$$

➡️ **Важно в задачах, где критично не пропустить положительный случай (например, обнаружение мошенничества).**

---

#### ⚖️ **4. F1-score (Гармоническое среднее Precision и Recall)**
$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

➡️ **Используется, когда нужно учитывать баланс между Precision и Recall.**


In [ ]:
from sklearn.metrics import classification_report

## Выведите classification_report
...

## ROC AUC SCORE

### ROC-кривая
1. Сортируем предсказанные вероятности по убыванию.
2. Для каждого значения берем его как **порог** и пересчитываем **TPR** и **FPR**:
- **TPR (True Positive Rate)** = $$\frac{TP}{(TP + FN)}$$  
- **FPR (False Positive Rate)** = $$\frac{FP}{(FP + TN)}$$
4. Строим график **FPR (ось X) vs TPR (ось Y)**.
5. Добавляем две крайние точки:
   - (0,0) — если порог максимальный, всё предсказано как 0.
   - (1,1) — если порог минимальный, всё предсказано как 1.
<figure>
    <img src="https://share.tinkoff.ru/s/6Qd9t7nKenkrSLk/download?path=&files=" alt="ROC CURVE"  width="300"/>
    <figcaption>ROC CURVE</figcaption>
</figure>

**ROC AUC SCORE** - это площадь под `ROC-кривой`

In [ ]:
from sklearn.metrics import ...

## Посчитайте roc auc, посмотрите в документации что использовать
print('ROC AUC SCORE:', ...)
print('-'*55)

In [ ]:
from sklearn.metrics import roc_curve


fpr, tpr, roc_thresholds = roc_curve(y_test, y_preds_proba)
auc_value = roc_auc_score(y_test, y_preds_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='orange', lw=2, label=f'ROC Curve (AUC = {auc_value:.2f})')
plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--', lw=1, label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve


precision, recall, pr_thresholds = precision_recall_curve(y_test, y_preds_proba)

plt.figure(figsize=(8, 6))
plt.plot(pr_thresholds, precision[:-1], color='gold', lw=2, label='Precision')
plt.plot(pr_thresholds, recall[:-1], color='red', lw=2, label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Threshold')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()